<a href="https://colab.research.google.com/github/emilymhudson/detection-as-code/blob/main/Detection_Engineering_Dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Generate the YARA rules

In [25]:
%%writefile advanced_threat_signatures.yara
/*
    Advanced Multimodal Threat Signatures
    Author: Emily Hudson
    Framework: MITRE ATT&CK (T1566.002, T1036.008)
    Description: Heuristic detection logic targeting generative AI media container
                 anomalies and advanced linguistic obfuscation tactics.
*/

rule Linguistic_Zero_Width_Obfuscation {
    meta:
        description = "Detects invisible zero-width unicode characters used to fragment strings and bypass Secure Email Gateways (SEGs)."
        severity = "High"
        confidence = "High"
        mitre_technique = "T1566.002 - Phishing: Spearphishing Link"

    strings:
        // Hex representations of zero-width space, non-joiner, joiner, and BOM
        $zw_space = { E2 80 8B }
        $zw_non_joiner = { E2 80 8C }
        $zw_joiner = { E2 80 8D }
        $zw_byte_order_mark = { EF BB BF }

        // Contextual anchors: common credential harvesting themes
        $anchor1 = "verify" ascii wide nocase
        $anchor2 = "password" ascii wide nocase
        $anchor3 = "aws" ascii wide nocase
        $anchor4 = "login" ascii wide nocase

    condition:
        any of ($anchor*) and 2 of ($zw_*)
}

rule Synthetic_Audio_Container_Anomaly {
    meta:
        description = "Identifies structural anomalies in RIFF/WAV headers lacking standard encoder metadata, indicative of raw synthetic generation (e.g., text-to-speech deepfakes)."
        severity = "Critical"
        confidence = "Medium"
        mitre_technique = "T1036.008 - Masquerading: Deepfake/Synthetic Media"

    strings:
        // RIFF header validation
        $riff_magic = { 52 49 46 46 }
        $wave_magic = { 57 41 56 45 }

        // Common human-recorded software encoder tags (Lavf, ProTools, Audacity)
        $standard_encoder1 = "Lavf" ascii
        $standard_encoder2 = "ProTools" ascii
        $standard_encoder3 = "Audacity" ascii

    condition:
        $riff_magic at 0 and $wave_magic at 8 and not any of ($standard_encoder*) and filesize < 5MB
}

Overwriting advanced_threat_signatures.yara


Statistical Threat Hunting Analytics

In [26]:
%%writefile ml_fusion_telemetry_hunting.spl
/*
    Splunk SPL: Statistical Outlier Detection for Multimodal Anomalies
    Author: Emily Hudson
    Description: Correlates JSON telemetry from the MIME Parser and Late Fusion Engine.
                 Identifies entities experiencing alert severities mathematically outside
                 their 30-day behavioral baseline (2+ standard deviations).
*/

index=enterprise_email sourcetype=ml_fusion_engine
(obfuscation_flags="ZERO_WIDTH_CHARACTER_INJECTION" OR fusion_calibrated_probability > 0.80)
| bin _time span=1d
| stats count as daily_threats max(fusion_calibrated_probability) as peak_threat_score by target_user, target_domain
| eventstats avg(daily_threats) as avg_threats stdev(daily_threats) as stdev_threats by target_domain
| eval threshold = avg_threats + (stdev_threats * 2)
| where daily_threats > threshold AND peak_threat_score >= 0.85
| eval alert_context = "User " + target_user + " targeted by sophisticated campaign exceeding 95th percentile historical baseline."
| table _time, target_user, daily_threats, threshold, peak_threat_score, alert_context
| sort - peak_threat_score

Overwriting ml_fusion_telemetry_hunting.spl


CI/CD Pipeline Validation Script

In [27]:
%%writefile validate_rules.py
import yara
import os
import sys

def compile_and_test_yara(rule_path):
    print(f"[*] Compiling YARA rule set: {rule_path}")
    try:
        # Compile the rules
        rules = yara.compile(filepath=rule_path)
        print("[+] Compilation Successful. Zero syntax errors detected.")

        # Inject two different zero-width characters
        simulated_payload = b"Please v" + b"\xe2\x80\x8b" + b"erify your A" + b"\xe2\x80\x8c" + b"WS login details."

        print("\n[*] Injecting simulated adversarial payload into memory...")
        matches = rules.match(data=simulated_payload)

        if matches:
            print(f"[!] SUCCESS: Payload intercepted. Rule triggered: {matches[0].rule}")
            for string_match in matches[0].strings:
                print(f"    -> Matched Byte Sequence: {string_match.identifier}")
        else:
            print("[-] FAILED: Rule did not trigger on simulated payload.")
            sys.exit(1)

    except yara.SyntaxError as e:
        print(f"[X] FATAL: YARA Syntax Error - {e}")
        sys.exit(1)

if __name__ == "__main__":
    if not os.path.exists("advanced_threat_signatures.yara"):
        print("[X] FATAL: Signature file not found.")
        sys.exit(1)

    print("==================================================")
    print("    DETECTION-AS-CODE CI/CD VALIDATION ENGINE     ")
    print("==================================================")
    compile_and_test_yara("advanced_threat_signatures.yara")

Overwriting validate_rules.py


In [28]:
!python validate_rules.py

    DETECTION-AS-CODE CI/CD VALIDATION ENGINE     
[*] Compiling YARA rule set: advanced_threat_signatures.yara
[+] Compilation Successful. Zero syntax errors detected.

[*] Injecting simulated adversarial payload into memory...
[!] SUCCESS: Payload intercepted. Rule triggered: Linguistic_Zero_Width_Obfuscation
    -> Matched Byte Sequence: $zw_space
    -> Matched Byte Sequence: $zw_non_joiner
    -> Matched Byte Sequence: $anchor4
